# W08. Uczenie maszynowe w psychologii

Psychoinformatyka • wersja 1.2.7 • 2026-11-04
Wymagania: Python >= 3.11, NumPy, pandas, SciPy, scikit-learn
Przykład przewidywania syntetycznej etykiety podwyższonego stresu dla nowych uczestników

## 1. Syntetyczne dane i etykiety

In [ ]:
import numpy as np
import pandas as pd
from scipy.special import expit
rng = np.random.default_rng(42)
pid = np.repeat(np.arange(1, 121), 4)
n = len(pid)
intercept = rng.normal(0, 0.55, 120)
sen = np.clip(rng.normal(7.1, 1.0, n), 4, 10)
telefon = np.clip(rng.normal(180, 75, n), 15, 500)
kroki = np.clip(rng.normal(7200, 2300, n), 500, 18000)
logit = (-0.4 - 0.85*(sen-7) + 0.004*(telefon-180)
         - 0.00012*(kroki-7000) + intercept[pid-1]
         + rng.normal(0, 0.7, n))
y = rng.binomial(1, expit(logit))
df = pd.DataFrame({'uczestnik':pid, 'sen_h':sen,
    'telefon_min':telefon, 'kroki':kroki, 'stres_wysoki':y})
df.loc[rng.choice(n, 20, replace=False), 'sen_h'] = np.nan


In [ ]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
X = df[['sen_h', 'telefon_min', 'kroki']]
y = df['stres_wysoki']
groups = df['uczestnik']
split = GroupShuffleSplit(n_splits=1, test_size=.25,
    random_state=42)
train, test = next(split.split(X, y, groups))
assert not (set(groups.iloc[train]) & set(groups.iloc[test]))
model = make_pipeline(SimpleImputer(strategy='median'),
    StandardScaler(), LogisticRegression(max_iter=1000))
baseline = DummyClassifier(strategy='most_frequent')


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score)
cv = GroupKFold(n_splits=5)
cv_bal = cross_val_score(model, X.iloc[train], y.iloc[train],
    groups=groups.iloc[train], cv=cv, scoring='balanced_accuracy')
print('CV (balanced accuracy):', np.round(cv_bal, 3))
for name, est in [('Bazowy', baseline), ('Logistyczny', model)]:
    est.fit(X.iloc[train], y.iloc[train])
    pred = est.predict(X.iloc[test])
    print(name, 'accuracy:', round(accuracy_score(y.iloc[test], pred), 3),
          'balanced:', round(balanced_accuracy_score(y.iloc[test], pred), 3))


CV (balanced accuracy): [0.615 0.624 0.552 0.684 0.555]
Bazowy accuracy: 0.575 balanced: 0.5
Logistyczny accuracy: 0.667 balanced: 0.623


In [ ]:
from sklearn.metrics import roc_curve
proba = model.predict_proba(X.iloc[test])[:, 1]
pred = (proba >= .5).astype(int)  # próg ustalony przed oceną testową
tn, fp, fn, tp = confusion_matrix(y.iloc[test], pred,
    labels=[0, 1]).ravel()
print('TN, FP, FN, TP:', tn, fp, fn, tp)
print('Precision:', round(precision_score(y.iloc[test], pred), 3),
      'Recall:', round(recall_score(y.iloc[test], pred), 3),
      'F1:', round(f1_score(y.iloc[test], pred), 3))
print('ROC-AUC:', round(roc_auc_score(y.iloc[test], proba), 3),
      'PR-AUC:', round(average_precision_score(y.iloc[test], proba), 3))
fpr, tpr, _ = roc_curve(y.iloc[test], proba)


TN, FP, FN, TP: 63 6 34 17
Precision: 0.739 Recall: 0.333 F1: 0.459
ROC-AUC: 0.75 PR-AUC: 0.726


In [ ]:
from sklearn.metrics import brier_score_loss
print("Brier:", round(brier_score_loss(y.iloc[test], proba), 3))
for prog in (0.3, 0.5, 0.7):
    etykiety = (proba >= prog).astype(int)
    prec = precision_score(y.iloc[test], etykiety, zero_division=0)
    rec = recall_score(y.iloc[test], etykiety)
    print("Próg:", prog, "precision:", round(prec, 3),
          "recall:", round(rec, 3))


Brier: 0.206
Próg: 0.3 precision: 0.544 recall: 0.725
Próg: 0.5 precision: 0.739 recall: 0.333
Próg: 0.7 precision: 1.0 recall: 0.078
